In [1]:
import sys
from pathlib import Path

# Add project root to sys.path
sys.path.append(str(Path.cwd().parent))

In [2]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from src.preprocessor import TVAEPreprocessorConfig, TVAEPreprocessor
from src.inference import reconstruct, generate_synthetic_samples
from src.train import train_tvae_model, run_latent_dim_sweep
from src.model import TVAEConfig, set_seed
from src.diagnostics import compute_discrete_class_weights

In [3]:
# Seed once, globally.
set_seed(42)

housing = fetch_california_housing(as_frame=True)
data = housing.frame

In [4]:
real_train_df, real_holdout_df = train_test_split(data, test_size=0.2, random_state=42)
real_train_df = real_train_df.reset_index(drop=True)
real_holdout_df = real_holdout_df.reset_index(drop=True)
print(f"train={len(real_train_df)}, holdout={len(real_holdout_df)}")

train=16512, holdout=4128


In [5]:
# Preprocess: n_gmm_components=5 matches the notebook (9 cols * (5+1) = 54 = input_dim)
config = TVAEPreprocessorConfig(n_gmm_components=5)
preprocessor = TVAEPreprocessor(config)

all_cols = list(data.columns)
transformed_train = preprocessor.fit_transform(real_train_df, continuous_columns=all_cols, categorical_columns=[])
transformed_holdout = preprocessor.transform(real_holdout_df)

input_dim = preprocessor.get_output_dim()
output_info = preprocessor.get_output_info()

class_weights = compute_discrete_class_weights(transformed_train, output_info)

print(f"input_dim={input_dim}, transformed shape={transformed_train.shape}")

input_dim=54, transformed shape=(16512, 54)


In [6]:
# Loaders 80/20 split
dataset = TensorDataset(torch.FloatTensor(transformed_train))
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = torch.utils.data.random_split(
    dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42)
)

BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

In [7]:
base_sweep_config = TVAEConfig(
    input_dim=input_dim,
    output_info=output_info,
    num_epochs=30,
    beta=1, beta_start=0.5, beta_warmup_epochs=6, use_beta_warmup=False,
    batch_size=BATCH_SIZE,
    device="cpu",
    dropout_rate=0.1
)

best_latent_dim, sweep_df = run_latent_dim_sweep(
    base_sweep_config, candidate_dims=[4, 8, 12, 16],
    train_loader=train_loader, val_loader=val_loader, verbose=True,
)

Epoch 10/30 - Train Loss: -2.0393 | Val Loss: -2.5067
Epoch 20/30 - Train Loss: -2.2619 | Val Loss: -2.7116
Epoch 30/30 - Train Loss: -2.3341 | Val Loss: -2.8311

Restored best checkpoint (prior_mismatch=0.2006)
latent_dim=4 -> final val_loss=-2.8311
Epoch 10/30 - Train Loss: -1.8782 | Val Loss: -2.3780
Epoch 20/30 - Train Loss: -2.1740 | Val Loss: -2.6271
Epoch 30/30 - Train Loss: -2.3030 | Val Loss: -2.7774

Restored best checkpoint (prior_mismatch=0.5415)
latent_dim=8 -> final val_loss=-2.7774
Epoch 10/30 - Train Loss: -1.6115 | Val Loss: -1.9719
Epoch 20/30 - Train Loss: -2.1266 | Val Loss: -2.6146
Epoch 30/30 - Train Loss: -2.2903 | Val Loss: -2.7315

Restored best checkpoint (prior_mismatch=0.6355)
latent_dim=12 -> final val_loss=-2.7315
Epoch 10/30 - Train Loss: -1.5152 | Val Loss: -1.8525
Epoch 20/30 - Train Loss: -2.1179 | Val Loss: -2.5595
Epoch 30/30 - Train Loss: -2.2961 | Val Loss: -2.7374

Restored best checkpoint (prior_mismatch=0.6820)
latent_dim=16 -> final val_loss=-2

In [8]:
final_config = TVAEConfig(
    input_dim=input_dim,
    output_info=output_info,
    latent_dim=best_latent_dim,
    num_epochs=100,
    beta=1, beta_start=0.5, beta_warmup_epochs=20, use_beta_warmup=False,
    batch_size=BATCH_SIZE,
    device="cpu",
    dropout_rate=0.1
)

model, trainer, history = train_tvae_model(final_config, train_loader, val_loader, verbose=True)

Epoch 10/100 - Train Loss: -2.0393 | Val Loss: -2.5067
Epoch 20/100 - Train Loss: -2.2619 | Val Loss: -2.7116
Epoch 30/100 - Train Loss: -2.3341 | Val Loss: -2.8311
Epoch 40/100 - Train Loss: -2.3922 | Val Loss: -2.9515
Epoch 50/100 - Train Loss: -2.4683 | Val Loss: -3.0237
Epoch 60/100 - Train Loss: -2.4849 | Val Loss: -3.0435
Epoch 70/100 - Train Loss: -2.5182 | Val Loss: -3.0663
Epoch 80/100 - Train Loss: -2.5494 | Val Loss: -3.1514
Epoch 90/100 - Train Loss: -2.5792 | Val Loss: -3.0754
Epoch 100/100 - Train Loss: -2.5989 | Val Loss: -3.1510

Restored best checkpoint (prior_mismatch=0.1503)
latent_dim=4 -> final val_loss=-3.1510


In [9]:
reconstructed_transformed = reconstruct(model, transformed_train, "cpu", output_info)
mse_recon = np.mean((transformed_train - reconstructed_transformed) ** 2)
reconstructed_data = preprocessor.inverse_transform(reconstructed_transformed)
print(f"VAE Reconstruction MSE: {mse_recon:.6f}")
reconstructed_data.head()

VAE Reconstruction MSE: 13.285412


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,3.509339,16.830314,4.842705,1.059084,1625.590765,3.027019,33.315213,-117.187548,1.597770
1,3.655144,36.054101,4.636433,1.049877,796.402982,2.301409,34.012827,-118.235925,3.623205
2,3.592393,16.423922,5.114732,1.075191,1654.450807,3.018292,34.192427,-120.834099,1.585660
3,1.950139,17.315029,4.255006,1.056345,1593.821661,3.380606,33.147764,-117.035192,0.863431
4,3.558483,16.464380,5.090629,1.060875,858.241568,2.839380,36.698482,-119.827619,0.847108


In [10]:
n_synthetic = len(real_train_df)
synthetic_transformed = generate_synthetic_samples(model, n_synthetic, "cpu", output_info)
synthetic_data = preprocessor.inverse_transform(synthetic_transformed)

for col in data.columns:
    print(f"  {col}: Original={data[col].mean():.4f}, Synthetic={synthetic_data[col].mean():.4f}")

  MedInc: Original=3.8707, Synthetic=3.8600
  HouseAge: Original=28.6395, Synthetic=29.1232
  AveRooms: Original=5.4290, Synthetic=5.2969
  AveBedrms: Original=1.0967, Synthetic=1.0602
  Population: Original=1425.4767, Synthetic=1081.7647
  AveOccup: Original=3.0707, Synthetic=2.8963
  Latitude: Original=35.6319, Synthetic=35.5475
  Longitude: Original=-119.5697, Synthetic=-119.4642
  MedHouseVal: Original=2.0686, Synthetic=1.9827


In [11]:
from sdmetrics.reports.single_table import QualityReport, DiagnosticReport
from sdv.metadata import SingleTableMetadata

In [12]:
metadata = SingleTableMetadata()

metadata.detect_from_dataframe(data)

print("Detected metadata:")
print(metadata.to_dict())

Detected metadata:
{'METADATA_SPEC_VERSION': 'SINGLE_TABLE_V1', 'columns': {'MedInc': {'sdtype': 'numerical'}, 'HouseAge': {'sdtype': 'numerical'}, 'AveRooms': {'sdtype': 'numerical'}, 'AveBedrms': {'sdtype': 'numerical'}, 'Population': {'sdtype': 'numerical'}, 'AveOccup': {'sdtype': 'numerical'}, 'Latitude': {'pii': True, 'sdtype': 'latitude'}, 'Longitude': {'pii': True, 'sdtype': 'longitude'}, 'MedHouseVal': {'sdtype': 'numerical'}}}


In [13]:
report = QualityReport()
report.generate(data, synthetic_data, metadata.to_dict())

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 135.17it/s]|
Column Shapes Score: 77.12%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 514.37it/s]|
Column Pair Trends Score: 87.69%

Overall Score (Average): 82.4%



In [14]:
diagnostioc_report = DiagnosticReport()
diagnostioc_report.generate(data, synthetic_data, metadata.to_dict())

Generating report ...

(1/2) Evaluating Data Validity: |██████████| 9/9 [00:00<00:00, 709.79it/s]|
Data Validity Score: 100.0%

(2/2) Evaluating Data Structure: |██████████| 1/1 [00:00<00:00, 727.17it/s]|
Data Structure Score: 100.0%

Overall Score (Average): 100.0%



In [15]:
from sdv.metadata import Metadata

metadata = Metadata.detect_from_dataframes(data = {
    'real_data': data,
})

metadata.to_dict()

{'tables': {'real_data': {'columns': {'MedInc': {'sdtype': 'numerical'},
    'HouseAge': {'sdtype': 'numerical'},
    'AveRooms': {'sdtype': 'numerical'},
    'AveBedrms': {'sdtype': 'numerical'},
    'Population': {'sdtype': 'numerical'},
    'AveOccup': {'sdtype': 'numerical'},
    'Latitude': {'pii': True, 'sdtype': 'latitude'},
    'Longitude': {'pii': True, 'sdtype': 'longitude'},
    'MedHouseVal': {'sdtype': 'numerical'}}}},
 'relationships': [],
 'METADATA_SPEC_VERSION': 'V1'}

In [16]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score

X_train = real_train_df.drop(columns=['MedHouseVal'])
y_train = real_train_df['MedHouseVal']

X_syn = synthetic_data.drop(columns=['MedHouseVal'])
y_syn = synthetic_data['MedHouseVal']

X_test = real_holdout_df.drop(columns=['MedHouseVal'])
y_test = real_holdout_df['MedHouseVal']

m_real = GradientBoostingRegressor()
m_real.fit(X_train, y_train)
r2_real = r2_score(y_test, m_real.predict(X_test))

m_syn = GradientBoostingRegressor()
m_syn.fit(X_syn, y_syn)
r2_syn = r2_score(y_test, m_syn.predict(X_test))

results = []
results.append({
    'r2_real': r2_real,
    'r2_synthetic': r2_syn,
    'utility_ratio': r2_syn / r2_real
})

utility_df = pd.DataFrame(results)
print(utility_df)

    r2_real  r2_synthetic  utility_ratio
0  0.775645      0.555221       0.715819
